# 00 — Setup e Qualidade de Dados

**Objetivo**: carregar `trusted_municipios`, validar estrutura e gerar o relatório de Data Quality da Etapa 2.

**Inputs**: `trusted_municipios` (BigQuery).

**Outputs**:
- `data/processed/reports/data_quality_report.json`
- `data/processed/figures/00_missing_values_heatmap.png`
- `data/processed/figures/00_municipios_por_uf.png`
- `data/processed/figures/00_populacao_total_distribuicao.png`

In [ ]:
# Imports e configuração
import logging

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.utils.bigquery import read_table_to_dataframe
from src.utils.eda import (
    compute_descriptive_stats,
    compute_null_profile,
    plot_distribution,
    save_figure,
    save_json,
)

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

In [ ]:
# Constantes
OUTPUT_REPORT = "data_quality_report.json"

## 1. Leitura da base trusted

In [ ]:
df = read_table_to_dataframe("trusted_municipios")
logger.info("Linhas: %d, Colunas: %d", df.shape[0], df.shape[1])
df.head()

## 2. Volumetria e integridade

In [ ]:
assert df.shape[0] == 5570, f"Esperado 5570 municípios, obtido {df.shape[0]}"
assert df["id_municipio"].nunique() == df.shape[0], "id_municipio duplicado"
assert df["id_municipio"].isnull().sum() == 0, "id_municipio nulo"

logger.info("Volumetria OK: %d municípios", df.shape[0])
logger.info("Municípios únicos: %d", df["id_municipio"].nunique())

## 3. Perfil de nulos

In [ ]:
null_profile = compute_null_profile(df)
display(null_profile)

In [ ]:
# Heatmap de missing values
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap="viridis", ax=ax)
ax.set_title("Mapa de Missing Values")
save_figure(fig, "00_missing_values_heatmap.png")
plt.show()

## 4. Estatísticas descritivas

In [ ]:
desc = compute_descriptive_stats(df)
display(desc)

## 5. Validação de regras de negócio

In [ ]:
rules = {
    "populacao_total_positiva": bool((df["populacao_total"] > 0).all()),
    "pib_per_capita_positivo": bool((df["pib_per_capita"] > 0).all()),
    "percentuais_entre_0_100": bool(
        df[["populacao_18_35_pct", "populacao_urbana_pct", "escolaridade_ensino_medio_pct"]]
        .apply(lambda s: s.between(0, 100).all())
        .all()
    ),
    "agencias_nao_negativas": bool((df["quantidade_agencias"].fillna(0) >= 0).all()),
}

for rule, ok in rules.items():
    logger.info("%s: %s", rule, "PASS" if ok else "FAIL")

assert all(rules.values()), "Regra de negócio falhou"

## 6. Diagnóstico de gaps conhecidos

In [ ]:
gaps = {
    "domicilios_com_internet_pct_nulos_pct": float(
        df["domicilios_com_internet_pct"].isnull().mean() * 100
    ),
    "estban_nulos_pct": float(df["quantidade_agencias"].isnull().mean() * 100),
    "idhm_vintage": "2010",
}
logger.info("Gaps conhecidos: %s", gaps)

## 7. Visualizações iniciais

In [ ]:
# Distribuição da população total (escala log)
plot_distribution(
    df, "populacao_total", log_scale=True, filename="00_populacao_total_distribuicao.png"
)
plt.show()

In [ ]:
# Municípios por UF
fig, ax = plt.subplots(figsize=(14, 6))
uf_counts = df["sigla_uf"].value_counts().sort_values(ascending=True)
uf_counts.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Quantidade de Municípios por UF")
ax.set_xlabel("Municípios")
save_figure(fig, "00_municipios_por_uf.png")
plt.show()

## 8. Salvamento do relatório

In [ ]:
report = {
    "volumetria": int(df.shape[0]),
    "colunas": int(df.shape[1]),
    "nulos_por_coluna": null_profile.set_index("coluna").to_dict(),
    "regras_negocio": rules,
    "gaps_conhecidos": gaps,
}
save_json(report, OUTPUT_REPORT)